# NB2d — AI Counter-Generation: DeepSeek

First of my six generation notebooks, rewritten for a **paid production run** where losing an
article means losing money. Every safeguard here exists so a crash, timeout, or API error never
costs me a generated article or makes me pay for it twice.

**Input:** `fact_cards.parquet` (NB2c, rebuilt on the new un-sanitized corpus).
**Output:** `aig_deepseek.parquet` — up to 900 AI articles, each paired to a human original.

## Safety guarantees (this run costs real money)

- **Save after EVERY article.** If the kernel dies at article 617, 616 is already on disk and I
  resume from 617.
- **Print EVERY article** the moment it finishes, so I can watch live and abort instantly.
- **A summary line every 10 articles.**
- **No exception ever loses an article.** Every article is wrapped in try/except; a failure is
  written with `error` set and the loop continues.
- **Fully resumable.** On restart it reads what's saved and generates only what's missing.

## Two phases

- **Phase A — pilot, 30 cards, synchronous.** Inspect quality/gate/length/cost before spending.
- **Phase B — production, 900 cards.** Same safeguards, set `PILOT_ONLY = False`.

## Prompt policy

Strong, realistic journalism. I never tune the prompt against my five statistical detection
features — that would make the detector circular. The model is never told this text is for
detection.

## Setup, secrets, config

In [1]:
!pip -q install openai >/dev/null 2>&1

import pandas as pd, numpy as np, json, re, os, time, random
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# DeepSeek is OpenAI-API-compatible. The secret MUST be attached to THIS notebook
# (Add-ons -> Secrets). A fresh notebook does not inherit secrets automatically.
API_KEY = UserSecretsClient().get_secret('DEEPSEEK_API_KEY')
client   = OpenAI(api_key=API_KEY, base_url='https://api.deepseek.com')

MODEL_NAME   = 'deepseek-chat'
GENERATOR_ID = 'deepseek'
N_TARGET     = 900
CARDS_PATH   = '/kaggle/input/notebooks/bahaaqassem/nb2c-build-fact-cards/fact_cards.parquet'
OUT_DIR      = '/kaggle/working'

PILOT_N      = 30
PILOT_ONLY   = False          # True = 30-article pilot only. Set False for the 900 production run.

PRICE_IN, PRICE_OUT = 1.74, 3.48     # DeepSeek $/1M tokens, for cost projection

cards = pd.read_parquet(CARDS_PATH)
print('cards:', cards.shape)

cards: (3500, 16)


## Which cards this generator takes

Deterministic contiguous slice per generator (seeded shuffle then slice), so the six notebooks
never overlap and the assignment is reproducible.

In [2]:
ALLOCATION = [('gpt', 700), ('sonnet', 450), ('opus', 450),
              ('gemini', 400), ('deepseek', 900), ('qwen', 600)]   # = 3500

shuffled = cards.sample(frac=1.0, random_state=42).reset_index(drop=True)
start = 0
for gid, n in ALLOCATION:
    if gid == GENERATOR_ID:
        my_cards = shuffled.iloc[start:start + n].reset_index(drop=True)
        break
    start += n
print(f'{GENERATOR_ID}: {len(my_cards)} cards | sample ids:', my_cards['pair_id'].head(3).tolist())

deepseek: 900 cards | sample ids: ['HA_03290', 'HA_01711', 'HA_01339']


## Prompt (mirror rule + explicit length band + no-formatting)

Conditional requirements fire only when the source actually has that element (mirror rule). I state
an explicit length band and forbid markdown so the raw output already matches the human corpus
(single paragraph, no `**`).

In [3]:
STYLE_HINTS = [
    'ابدأ بفقرة استهلالية تلخّص الحدث',
    'اذكر خلفية موجزة للسياق',
    'أورد ردود فعل الأطراف المعنية',
    'اختم بما هو متوقّع أو منتظر',
    'استخدم بنية الهرم المقلوب',
    'انسب المعلومات إلى مصادرها',
]

SYSTEM_PROMPT = (
    'أنت صحفي محترف يكتب تقارير إخبارية بالعربية الفصحى لصالح غرفة أخبار محترمة. '
    'تكتب بأسلوب صحفي رصين ودقيق، وتلتزم بالحقائق المعطاة دون إضافة معلومات من خارجها.'
)

# ---- number cleanup so prompt & gate agree (dual-calendar months, junk single digits) ----
MONTH_GROUPS = [
    {'يناير','كانون الثاني'}, {'فبراير','شباط'}, {'مارس','آذار'},
    {'أبريل','نيسان'}, {'مايو','أيار'}, {'يونيو','حزيران'},
    {'يوليو','تموز'}, {'أغسطس','آب'}, {'سبتمبر','أيلول'},
    {'أكتوبر','تشرين الأول'}, {'نوفمبر','تشرين الثاني'}, {'ديسمبر','كانون الأول'},
]
AMBIG = {'كانون': 0, 'تشرين': 9}

def month_group(item):
    for i, g in enumerate(MONTH_GROUPS):
        if any(m in item for m in g):
            return i
    for k, i in AMBIG.items():
        if k in item:
            return i
    return None

def clean_numbers(nums, cap=6):
    seen, out = set(), []
    for n in nums:
        n = n.strip()
        if re.fullmatch(r'[٠-٩0-9]', n):     # bare single digit = list marker
            continue
        g = month_group(n)
        if g is not None:
            if g in seen:
                continue
            seen.add(g)
        out.append(n)
    return out[:cap]

def build_prompt(card):
    ents   = json.loads(card['entities_for_prompt'])
    facts  = json.loads(card['fact_points'])
    target = int(card['target_words'])

    parts = [
        'اكتب تقريراً إخبارياً بالعربية الفصحى عن الموضوع التالي.',
        '',
        f'الموضوع: {card["topic_core"]}',
        '',
        'الكيانات التي يجب أن يذكرها التقرير:',
        '، '.join(ents),
        '',
        'الحقائق الأساسية التي يجب تغطيتها:',
    ]
    for f in facts:
        parts.append(f'- {f}')

    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            parts += ['', 'أدرج هذه الأرقام والتواريخ: ' + '، '.join(nums)]
    if card['has_agencies']:
        ags = json.loads(card['source_agencies'])
        parts += ['', 'انسب المعلومات إلى: ' + '، '.join(ags)]
    if card['has_quotes']:
        parts += ['', 'أدرج تصريحات منسوبة للأطراف المعنية، بصياغتك أنت.']

    hints = random.sample(STYLE_HINTS, k=random.choice([3, 4]))
    parts += ['', 'إرشادات التحرير:'] + [f'- {h}' for h in hints]
    parts += ['',
              f'الطول: لا يقل التقرير عن {target} كلمة ولا يزيد عن {int(target*1.12)} كلمة. '
              f'اكتب تقريراً مكتملاً ضمن هذا النطاق.',
              '',
              'اكتب نص التقرير مباشرة: دون عنوان، ودون أي تنسيق (لا نجوم ** ولا رموز تنسيق)، '
              'ودون مقدمة أو تعليق منك.']
    return '\n'.join(parts)

## Formatting normalization (shared by both classes)

The model tends to wrap articles in a bold headline and split them into paragraphs; my human
corpus has none of that. I strip formatting artifacts and collapse newlines so the raw AI output
already matches the human single-paragraph shape. NB3 applies the identical function to both
classes, keeping the treatment symmetric.

In [4]:
def normalize_format(text):
    t = str(text)
    t = re.sub(r'\*\*(.+?)\*\*', r'\1', t)               # bold
    t = re.sub(r'__(.+?)__', r'\1', t)
    t = re.sub(r'(?<!\w)\*(.+?)\*(?!\w)', r'\1', t)      # italic
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.M)          # headings
    t = re.sub(r'^\s*[-–—>]\s+', '', t, flags=re.M)   # bullets / quotes
    t = re.sub(r'^\s*[-*_]{3,}\s*$', '', t, flags=re.M)   # rules
    t = re.sub(r'\n+', ' ', t)                            # human text has no newlines
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()

## Decoding parameters and generation (measured length top-up)

Temperature 0.8–1.1 per article, top_p 0.95, light frequency penalty; all logged so any article is
reproducible. If a draft is clearly short (< 88% of target) I ask for a *measured* continuation so
I don't overshoot the band (the pilot's only length misses were slight overshoots).

In [5]:
def sample_params():
    return {
        'temperature':       round(random.uniform(0.8, 1.1), 3),
        'top_p':             0.95,
        'frequency_penalty': round(random.uniform(0.2, 0.5), 3),
        'presence_penalty':  0.1,
    }

def _call(messages, params, max_tokens, max_retries=4):
    # retry transient API/network errors with backoff; never lose an article to a blip
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME, messages=messages, max_tokens=max_tokens, **params)
            return resp.choices[0].message.content.strip(), resp.usage
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

def generate_article(card, params):
    prompt = build_prompt(card)
    target = int(card['target_words'])
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt}]
    text, usage = _call(msgs, params, int(target * 3.2) + 500)
    text = normalize_format(text)
    tin, tout = usage.prompt_tokens, usage.completion_tokens

    n = len(text.split())
    if n < target * 0.88:
        need = int((target - n) * 0.9)          # measured, to avoid overshoot
        msgs2 = msgs + [{'role': 'assistant', 'content': text},
                        {'role': 'user',
                         'content': f'أضف نحو {need} كلمة فقط لإكمال التقرير بنفس الأسلوب، '
                                    f'دون تكرار ما ورد ودون تنسيق. اكتب التتمة فقط.'}]
        try:
            ext, usage2 = _call(msgs2, params, int(need * 3.2) + 300)
            text = normalize_format(text + ' ' + normalize_format(ext))
            tin += usage2.prompt_tokens; tout += usage2.completion_tokens
        except Exception:
            pass
    return text, tin, tout

## Acceptance gate (fuzzy, prefix-tolerant, mirror-rule aware)

Accept only if the article anchors to the same event: weighted coverage >= 80% AND entity coverage
>= 65% (hard floor). Only elements the source actually had are scored. Matching normalizes clitics,
`ال`, diacritics, alef/ya/ta-marbuta and Arabic-Indic vs Western digits; a month is satisfied by
either calendar name.

In [6]:
_AR_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩',
                           '0123456789')

def _norm(s):
    s = re.sub(r'[\u064B-\u0652]', '', s)                # diacritics
    s = s.translate(_AR_DIGITS)
    return (s.replace('أ','ا').replace('إ','ا').replace('آ','ا')
             .replace('ة','ه').replace('ى','ي'))

def _norm_entity(e):
    e = _norm(e)
    e = re.sub(r'^[وفبكل]?ال', '', e)
    e = re.sub(r'^لل', '', e)
    e = re.sub(r'^[وفبكل](?=.{3,})', '', e)
    return e.strip()

def _present(item, tn, is_entity=True):
    n = _norm_entity(item) if is_entity else _norm(item)
    return len(n) >= 2 and n in tn

def _number_present(item, tn):
    g = month_group(item)
    if g is not None:
        names = MONTH_GROUPS[g] | {k for k, v in AMBIG.items() if v == g}
        return any(_norm(m) in tn for m in names)
    return _present(item, tn, is_entity=False)

def acceptance_gate(article, card, W_ENT=3, W_NUM=2, W_AG=1):
    tn = _norm(article)
    scores, weights, detail = [], [], {}

    ents = json.loads(card['entities_for_prompt'])
    if ents:
        hit = sum(_present(e, tn) for e in ents)
        ent_cov = hit / len(ents)
        detail['entities'] = f'{hit}/{len(ents)}'
        scores.append(ent_cov); weights.append(W_ENT)
    else:
        ent_cov = 1.0
        detail['entities'] = 'none'

    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            hit = sum(_number_present(n, tn) for n in nums)
            c = hit / len(nums); detail['numbers'] = f'{hit}/{len(nums)}'
            scores.append(c); weights.append(W_NUM)
    if card['has_agencies']:
        ags = json.loads(card['source_agencies'])
        hit = sum(_present(a, tn, False) for a in ags)
        c = hit / max(len(ags), 1); detail['agencies'] = f'{hit}/{len(ags)}'
        scores.append(c); weights.append(W_AG)

    weighted = sum(s * w for s, w in zip(scores, weights)) / max(sum(weights), 1)
    passed = (weighted >= 0.80) and (ent_cov >= 0.65)
    return {'passed': passed, 'weighted': round(weighted, 3),
            'entity_cov': round(ent_cov, 3), 'detail': detail}

def length_ok(article, target, tol=0.15):
    n = len(article.split())
    return abs(n - target) / target <= tol, n

## Phase A — pilot (30 cards, synchronous)

Generates 30, prints each live, saves after each, reports gate/length/cost and a formatting-artifact
check. Nothing here is destructive; I read the results before spending on the rest.

In [7]:
PILOT_CKPT = f'{OUT_DIR}/pilot_{GENERATOR_ID}.parquet'

pilot = my_cards.head(PILOT_N)
rows, t0 = [], time.time()
tok_in = tok_out = 0

print(f'--- pilot: {PILOT_N} articles ---', flush=True)
for i, card in pilot.iterrows():
    a0 = time.time()
    try:
        params = sample_params()
        text, tin, tout = generate_article(card, params)
        gate = acceptance_gate(text, card)
        lok, nwords = length_ok(text, int(card['target_words']))
        tok_in += tin; tok_out += tout
        rows.append({'pair_id': card['pair_id'], 'passed': gate['passed'],
                     'weighted': gate['weighted'], 'entity_cov': gate['entity_cov'],
                     'len_ok': lok, 'words': nwords, 'target': int(card['target_words']),
                     'detail': str(gate['detail']), 'text': text})
        pd.DataFrame(rows).to_parquet(PILOT_CKPT, index=False)          # save after every article
        flag = 'OK ' if (gate['passed'] and lok) else 'BAD'
        print(f'[{len(rows):2d}/{PILOT_N}] {flag} {card["pair_id"]} | '
              f'cov={gate["weighted"]:.0%} ent={gate["entity_cov"]:.0%} '
              f'| {nwords}w/{int(card["target_words"])}w ({nwords/int(card["target_words"]):.2f}) '
              f'| md={chr(34)+chr(34) in text} | {time.time()-a0:.0f}s | {gate["detail"]}',
              flush=True)
    except Exception as e:
        print(f'[{len(rows):2d}/{PILOT_N}] ERR {card["pair_id"]}: {type(e).__name__}: {str(e)[:120]}',
              flush=True)

p = pd.DataFrame(rows)
print(f'\n=== pilot summary ===')
print(f'generated  : {len(p)}/{PILOT_N} in {time.time()-t0:.0f}s')
print(f'gate pass  : {p["passed"].sum()}/{len(p)} ({100*p["passed"].mean():.0f}%)')
print(f'length ok  : {p["len_ok"].sum()}/{len(p)} ({100*p["len_ok"].mean():.0f}%)')
print(f'entity cov : mean {p["entity_cov"].mean():.0%} | min {p["entity_cov"].min():.0%}')
print(f'weighted   : mean {p["weighted"].mean():.0%}')
print(f'words/target: mean {(p["words"]/p["target"]).mean():.2f}')
cost = tok_in/1e6*PRICE_IN + tok_out/1e6*PRICE_OUT
print(f'cost       : ${cost:.3f} | projected {N_TARGET}: ${cost/max(len(p),1)*N_TARGET:.2f}')

n_md = int(p['text'].str.contains('**', regex=False).sum())
n_nl = int(p['text'].str.contains(chr(10), regex=False).sum())
print(f'\nmarkdown left: {n_md} | newlines left: {n_nl}   (both must be 0)')
if n_md or n_nl:
    print('!! formatting artifacts present — do NOT run production until fixed')

--- pilot: 30 articles ---
[ 1/30] OK  HA_03290 | cov=88% ent=87% | 2191w/2358w (0.93) | md=False | 61s | {'entities': '27/31', 'numbers': '6/6', 'agencies': '2/3'}
[ 2/30] OK  HA_01711 | cov=95% ent=90% | 635w/595w (1.07) | md=False | 18s | {'entities': '18/20', 'numbers': '1/1', 'agencies': '3/3'}
[ 3/30] OK  HA_01339 | cov=96% ent=92% | 510w/535w (0.95) | md=False | 13s | {'entities': '24/26', 'numbers': '6/6', 'agencies': '2/2'}
[ 4/30] OK  HA_02897 | cov=86% ent=88% | 1180w/1070w (1.10) | md=False | 29s | {'entities': '15/17', 'numbers': '6/6', 'agencies': '1/2'}
[ 5/30] OK  HA_00453 | cov=96% ent=91% | 432w/429w (1.01) | md=False | 11s | {'entities': '10/11', 'numbers': '6/6', 'agencies': '1/1'}
[ 6/30] BAD HA_00913 | cov=80% ent=93% | 496w/472w (1.05) | md=False | 13s | {'entities': '14/15', 'numbers': '1/2', 'agencies': '2/2'}
[ 7/30] OK  HA_02359 | cov=98% ent=96% | 679w/728w (0.93) | md=False | 21s | {'entities': '21/22', 'numbers': '6/6', 'agencies': '2/2'}
[ 8/30] OK  HA_01

In [8]:
# read one full article + list any failures
if len(p):
    best = p.loc[p['weighted'].idxmax()]
    print('='*70); print('BEST', best['pair_id'], best['detail'], f"{best['words']}w/{best['target']}w")
    print('='*70); print(best['text'][:1200])
    fails = p[~p['passed']]
    print(f'\nfailed: {len(fails)}')
    for _, f in fails.iterrows():
        print(f"  {f['pair_id']}: w={f['weighted']:.0%} ent={f['entity_cov']:.0%} {f['detail']}")

BEST HA_00037 {'entities': '23/23', 'numbers': '6/6', 'agencies': '1/1'} 428w/424w
أكدت حركة فتح، اليوم، أن ما تقوم به حركة حماس من جباية مالية تحت مسميات ضريبية في قطاع غزة يعد عملاً غير قانوني، ويستهدف المواطنين المنهكين اقتصادياً في القطاع المحاصر. جاء ذلك على لسان عضو المجلس الثوري لحركة فتح، محمد اللحام، خلال اجتماع عقد في مقر إقليم فتح في بيت لحم، ضم مجموعة من كادر الحركة. وقال اللحام، وفق ما نقلته وكالة الأنباء الفلسطينية الرسمية (وفا)، إن الرئيس محمود عباس (أبو مازن) سبق وأصدر مرسوماً في مارس من العام 2007 يعفي بموجبه أهالي قطاع غزة من الضرائب، مشيراً إلى أن هذه الخطوة جاءت للتخفيف عن السكان في ظل الظروف الصعبة التي يعيشونها. وأضاف اللحام أن "حماس تستقوي بالبطش والجرائم على التجار والناس في سلبهم أرزاقهم"، في إشارة إلى الإجراءات التي تتخذها الحركة في القطاع. ويأتي هذا التصريح في وقت يشهد فيه قطاع غزة أزمة اقتصادية حادة، تفاقمت بفعل الحصار الإسرائيلي المستمر منذ العام 2007، والذي أعقب سيطرة حركة حماس على القطاع. وتشير تقديرات محلية إلى أن نسبة البطالة في غزة تجاوزت 37 في المئة، 

## Phase B — production (900 cards)

Set `PILOT_ONLY = False` and run. **Saves after every article. Prints every article. Summary every
10. Resumable. No exception loses an article.** A gate-failed article is written with
`gate_passed=False` (filtered, not silently dropped, in NB2j) so I never pay to regenerate blindly.
Across `MAX_ATTEMPTS` I keep the best attempt, so a retry never replaces a better draft with a worse
one.

In [9]:
def make_record(card, text, params, gate, nwords, lok, attempt, error=''):
    return {
        'id': f'AI_{GENERATOR_ID}_{card["pair_id"]}',
        'text': text, 'label': 'ai', 'generator': GENERATOR_ID,
        'source_pair_id': card['pair_id'],
        'temperature': params.get('temperature', 0), 'top_p': params.get('top_p', 0),
        'frequency_penalty': params.get('frequency_penalty', 0),
        'presence_penalty': params.get('presence_penalty', 0),
        'target_words': int(card['target_words']), 'actual_words': nwords,
        'entities_injected': card['entities_for_prompt'],
        'coverage_weighted': gate.get('weighted', 0), 'coverage_entities': gate.get('entity_cov', 0),
        'gate_passed': bool(gate.get('passed', False)), 'length_ok': bool(lok),
        'generation_attempts': attempt, 'error': error,
    }

In [10]:
CKPT         = f'{OUT_DIR}/aig_{GENERATOR_ID}.parquet'
MAX_ATTEMPTS = 2
RESUME_PATH  = None     # e.g. '/kaggle/input/aig-deepseek-partial/aig_deepseek.parquet'

if not PILOT_ONLY:
    done_ids, records = set(), []
    for pth in [RESUME_PATH, CKPT]:
        if pth and os.path.exists(pth):
            prev = pd.read_parquet(pth)
            records = prev.to_dict('records'); done_ids = set(prev['source_pair_id'])
            print(f'resuming: {len(done_ids)} already done', flush=True)
            break

    todo = my_cards[~my_cards['pair_id'].isin(done_ids)].reset_index(drop=True)
    print(f'total {len(my_cards)} | done {len(done_ids)} | remaining {len(todo)}', flush=True)

    t0 = time.time(); n_ok = n_rej = n_err = 0; tin_tot = tout_tot = 0
    for i, card in todo.iterrows():
        try:
            best_rec, params = None, {}
            for attempt in range(1, MAX_ATTEMPTS + 1):
                params = sample_params()
                text, tin, tout = generate_article(card, params)
                tin_tot += tin; tout_tot += tout
                gate = acceptance_gate(text, card)
                lok, nwords = length_ok(text, int(card['target_words']))
                rec = make_record(card, text, params, gate, nwords, lok, attempt)
                if best_rec is None or (rec['gate_passed'], rec['coverage_weighted']) > \
                                       (best_rec['gate_passed'], best_rec['coverage_weighted']):
                    best_rec = rec                 # keep the best attempt
                if gate['passed'] and lok:
                    break
            rec = best_rec
            n_ok += 1
            if not rec['gate_passed']:
                n_rej += 1
        except Exception as e:
            rec = make_record(card, '', {}, {}, 0, False, 0, f'{type(e).__name__}: {str(e)[:150]}')
            n_err += 1
            print(f'ERR {card["pair_id"]}: {rec["error"]}', flush=True)

        records.append(rec)
        pd.DataFrame(records).to_parquet(CKPT, index=False)          # SAVE AFTER EVERY ARTICLE

        done = len(records)
        status = 'ERR' if rec['error'] else ('OK ' if (rec['gate_passed'] and rec['length_ok']) else 'BAD')
        rate = (done - len(done_ids)) / max(time.time() - t0, 1)
        eta  = (len(todo) - (i + 1)) / max(rate, 1e-6) / 60
        print(f'[{done:4d}/{len(my_cards)}] {status} {card["pair_id"]} | '
              f'cov={rec["coverage_weighted"]:.0%} ent={rec["coverage_entities"]:.0%} '
              f'| {rec["actual_words"]}w/{rec["target_words"]}w | att={rec["generation_attempts"]} '
              f'| ok={n_ok} bad={n_rej} err={n_err} | ETA {eta:.0f}m', flush=True)

        if done % 10 == 0:
            cost = tin_tot/1e6*PRICE_IN + tout_tot/1e6*PRICE_OUT
            passpct = 100*(n_ok - n_rej)/n_ok if n_ok else 0
            print(f'    >>> {done}/{len(my_cards)} | pass {passpct:.0f}% | cost so far ${cost:.2f}',
                  flush=True)

    cost = tin_tot/1e6*PRICE_IN + tout_tot/1e6*PRICE_OUT
    print(f'\nDONE {GENERATOR_ID} | ok={n_ok} rejected={n_rej} err={n_err} | total cost ${cost:.2f}',
          flush=True)
else:
    print('PILOT_ONLY is True — production skipped. Review the pilot, then set PILOT_ONLY = False.')

total 900 | done 0 | remaining 900
[   1/900] OK  HA_03290 | cov=92% ent=84% | 2320w/2358w | att=1 | ok=1 bad=0 err=0 | ETA 879m
[   2/900] OK  HA_01711 | cov=95% ent=90% | 622w/595w | att=1 | ok=2 bad=0 err=0 | ETA 560m
[   3/900] BAD HA_01339 | cov=96% ent=92% | 618w/535w | att=1 | ok=3 bad=0 err=0 | ETA 517m
[   4/900] OK  HA_02897 | cov=92% ent=94% | 1020w/1070w | att=2 | ok=4 bad=0 err=0 | ETA 604m
[   5/900] OK  HA_00453 | cov=100% ent=100% | 392w/429w | att=1 | ok=5 bad=0 err=0 | ETA 515m
[   6/900] OK  HA_00913 | cov=83% ent=100% | 469w/472w | att=1 | ok=6 bad=0 err=0 | ETA 457m
[   7/900] OK  HA_02359 | cov=91% ent=82% | 686w/728w | att=1 | ok=7 bad=0 err=0 | ETA 432m
[   8/900] OK  HA_01199 | cov=81% ent=96% | 593w/533w | att=1 | ok=8 bad=0 err=0 | ETA 406m
[   9/900] OK  HA_01924 | cov=92% ent=83% | 530w/582w | att=1 | ok=9 bad=0 err=0 | ETA 386m
[  10/900] OK  HA_01274 | cov=81% ent=95% | 490w/500w | att=1 | ok=10 bad=0 err=0 | ETA 366m
    >>> 10/900 | pass 100% | cost so 

In [11]:
# final verification (production only)
if not PILOT_ONLY:
    final = pd.read_parquet(CKPT)
    print('shape        :', final.shape)
    print('unique ids   :', final['source_pair_id'].nunique(), '/', len(final))
    print('errors       :', int((final['error'] != '').sum()))
    print('gate passed  :', int(final['gate_passed'].sum()), f"({100*final['gate_passed'].mean():.0f}%)")
    print('length ok    :', int(final['length_ok'].sum()), f"({100*final['length_ok'].mean():.0f}%)")
    print('mean coverage:', f"{final['coverage_weighted'].mean():.0%}")
    print('mean attempts:', f"{final['generation_attempts'].mean():.2f}")
    n_md = int(final['text'].str.contains('**', regex=False).sum())
    n_nl = int(final['text'].str.contains(chr(10), regex=False).sum())
    print('markdown/newlines left:', n_md, '/', n_nl, '(must be 0)')
    print('\nupload as aigt-aig-deepseek for NB2j')

shape        : (900, 18)
unique ids   : 900 / 900
errors       : 0
gate passed  : 878 (98%)
length ok    : 786 (87%)
mean coverage: 93%
mean attempts: 1.14
markdown/newlines left: 0 / 0 (must be 0)

upload as aigt-aig-deepseek for NB2j


## Notes

- **Every-article save** is the core money-protector: the checkpoint is rewritten after each
  article, so a crash costs at most the one in flight, and resume skips everything already saved.
- `source_pair_id` on every row keeps human/AI pairs together for NB3's pair-aware split.
- Gate-failed articles are written with `gate_passed=False`, never silently dropped — the rejection
  rate stays auditable and I don't pay to regenerate blindly.
- Across `MAX_ATTEMPTS` I keep the best attempt, so a retry never downgrades a good draft.
- DeepSeek stays synchronous even in production: Batch's async model breaks the regenerate loop and
  saves only ~$4 here. Batch is reserved for the pricier Opus/GPT notebooks.